# Practical 13 — LLM Limitations and Rigorous Evaluation

This notebook runs an **evaluation harness** that scores a model's answers to
finance questions against a gold QA set and reports *honest* metrics: accuracy
and token-F1, plus **calibration** — does the model's stated confidence track
whether it is actually right? The pipeline is **Score -> Calibrate -> Report**,
and its whole point is to surface the **confident-but-wrong** answers that raw
accuracy hides.

In the real practical you drive a Claude Code / Cline agent that picks the eval
set and interprets the numbers while deterministic tools do all the scoring.
Here we call those same reference tools (`tools/score.py`, `tools/calibration.py`)
directly, fully **offline**, on the bundled NovaCorp Inc. data — the
Colab-friendly counterpart to the agentic project.

In [ ]:
# This practical's tools are Python standard library only —
# nothing to install. (Colab already has Python ready to go.)

In [ ]:
# --- setup: make this practical's tools importable (works locally & in Colab) ---
import os, sys, pathlib, subprocess
PRACTICAL = "13-llm-limitations-evaluation"
def _locate():
    for base in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents]:
        if base.name == PRACTICAL and (base / "tools").is_dir():
            return base
        cand = base / "code" / "practicals" / PRACTICAL
        if cand.is_dir():
            return cand
    return None
root = _locate()
if root is None:
    if not pathlib.Path("llm-finance-book").exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/jfimbett/llm-finance-book.git"], check=True)
    root = pathlib.Path("llm-finance-book/code/practicals") / PRACTICAL
root = root.resolve()
sys.path.insert(0, str(root))
sys.path.insert(0, str(root / "tools"))
os.chdir(root)
print("Practical root:", root)

## Step 0 — Load the bundled eval set

Everything is offline. `data/` ships a 10-item gold finance QA set for the
fictional **NovaCorp Inc.** and two candidate models answering the same
questions:

- `candidates_overconfident.json` — asserts ~0.9 confidence on almost
  everything but gets only half right.
- `candidates_calibrated.json` — more accurate, and it *hedges* (low
  confidence) on the answers it gets wrong.

`load_gold` returns `{id: {id, question, answer}}`; `load_predictions` returns
`{id: {id, answer, confidence}}`.

In [ ]:
import json
from tools._common import load_gold, load_predictions

gold = load_gold("data/gold.json")
over = load_predictions("data/candidates_overconfident.json")
cal  = load_predictions("data/candidates_calibrated.json")

print(f"gold questions: {len(gold)}   overconfident preds: {len(over)}   calibrated preds: {len(cal)}")
print("\nSample gold item:")
print(json.dumps(gold["q1"], indent=2))
print("\nSame question, two models:")
print("  overconfident:", over["q1"])
print("  calibrated:   ", cal["q1"])

## Step 1 — Score answers (exact-match + token-F1)

`score_dataset(gold, preds)` produces two reproducible numbers per item that
replace grading by hand:

- `exact_match` — 1 if the candidate equals the gold answer after
  normalisation (lowercase, punctuation stripped except `%` and decimals, so
  `"US Dollars"` == `"us dollars"` but `"1.4 billion"` != `"1.14 billion"`).
- `f1` — SQuAD-style token-level F1 in [0, 1] for partial credit.

An item is *correct* when it is an exact match. We score the overconfident
model first.

In [ ]:
from tools.score import score_dataset, exact_match, token_f1

scored_over = score_dataset(gold, over)
print(f"n={scored_over['n']}  n_correct={scored_over['n_correct']}  "
      f"accuracy={scored_over['accuracy']}  mean_f1={scored_over['mean_f1']}\n")

print(f"{'id':>4} {'em':>3} {'f1':>6} {'conf':>5}  gold -> candidate")
for it in scored_over["items"]:
    print(f"{it['id']:>4} {it['exact_match']:>3} {it['f1']:>6} {it['confidence']:>5}  "
          f"{it['gold']!r} -> {it['candidate']!r}")

Note F1 gives partial credit where exact-match gives none — e.g. q5's
`"1.4 billion dollars"` vs gold `"1.14 billion dollars"` shares 2 of 3 tokens,
so `exact_match=0` but `f1` is around 0.67. Accuracy alone would throw that
nuance away.

## Step 2 — Calibrate: ECE, reliability bins, confident-wrong cases

`analyze(items, n_bins, threshold)` measures whether the stated confidence can
be trusted:

- `reliability_bins` — group predictions by confidence; a well-calibrated model
  has per-bin accuracy ~= confidence.
- `ece` (Expected Calibration Error) — the N-weighted average gap
  `|accuracy - confidence|` across bins. Lower is better; 0 is perfect.
- `confident_wrong` — items the model got wrong while claiming confidence >=
  threshold (default 0.8). These are the dangerous failures: not just wrong,
  but *sure*.

In [ ]:
from tools.calibration import analyze, expected_calibration_error, confident_wrong

rep_over = analyze(scored_over["items"], n_bins=10, threshold=0.8)
print(f"accuracy={rep_over['accuracy']}  mean_confidence={rep_over['mean_confidence']}  "
      f"ECE={rep_over['ece']}\n")

print("Confident-but-WRONG (confidence >= 0.8), most sure first:")
for f in rep_over["confident_wrong"]:
    print(f"  {f['id']}  conf={f['confidence']}  said {f['candidate']!r}, "
          f"truth {f['gold']!r}  ({f['question']})")

print("\nNon-empty reliability bins (lo, hi: count  conf -> acc  gap):")
for b in rep_over["bins"]:
    if b["count"]:
        print(f"  [{b['lo']}, {b['hi']}]: n={b['count']}  "
              f"conf={b['confidence']} -> acc={b['accuracy']}  gap={b['gap']}")

The overconfident model packs almost everything into the high-confidence bins,
yet only half of those answers are right — so the gap (and ECE) is large, and
five items are confidently wrong. High accuracy would not have warned us.

## Step 3 — Run the same harness on the *calibrated* model

Same Score -> Calibrate pipeline, different candidate file. The README's first
"thing to try" is exactly this comparison.

In [ ]:
scored_cal = score_dataset(gold, cal)
rep_cal = analyze(scored_cal["items"], n_bins=10, threshold=0.8)

print(f"{'model':>14}  {'acc':>5}  {'mean_f1':>7}  {'mean_conf':>9}  {'ECE':>6}  {'conf_wrong':>10}")
for name, sc, rp in [("overconfident", scored_over, rep_over),
                     ("calibrated",    scored_cal,  rep_cal)]:
    print(f"{name:>14}  {sc['accuracy']:>5}  {sc['mean_f1']:>7}  "
          f"{rp['mean_confidence']:>9}  {rp['ece']:>6}  {len(rp['confident_wrong']):>10}")

print("\nCalibrated model's confident-wrong list:", rep_cal["confident_wrong"])

The calibrated model is *more* accurate **and** has zero confident-wrong cases:
it hedges (low confidence) exactly on q3, q6, q10 — the ones it gets wrong — so
its ECE is far lower. Same harness, very different trustworthiness. That is the
lesson: accuracy alone hides whether a model knows when it is wrong.

## Try it — lower the confident-wrong threshold

From the README: drop the threshold from 0.8 to 0.6 and watch more items get
flagged. The overconfident model states confidence in the 0.85–0.95 range on
everything, so lowering to 0.6 still flags exactly its five wrong answers (none
sit between 0.6 and 0.8), while the calibrated model — which parks its wrong
answers at 0.10–0.20 confidence — stays clean even at 0.6.

In [ ]:
for thr in (0.8, 0.6, 0.5):
    fo = confident_wrong(scored_over["items"], threshold=thr)
    fc = confident_wrong(scored_cal["items"],  threshold=thr)
    print(f"threshold={thr}:  overconfident flags {sorted(f['id'] for f in fo)}   "
          f"calibrated flags {sorted(f['id'] for f in fc)}")

## Try it — edit a confidence and watch ECE move

Another README experiment: make the overconfident model claim **0.99** on a
wrong answer and re-run. We mutate an in-memory copy (the bundled file is left
untouched) and re-score, showing ECE rise as confidence drifts further from
reality.

In [ ]:
import copy
mutated = copy.deepcopy(scored_over["items"])
for it in mutated:
    if it["id"] == "q3":          # q3 is wrong; push its confidence up
        before = it["confidence"]
        it["confidence"] = 0.99

ece_before = expected_calibration_error(scored_over["items"])
ece_after  = expected_calibration_error(mutated)
print(f"q3 confidence {before} -> 0.99 (still wrong)")
print(f"ECE before = {round(ece_before, 4)}   ECE after = {round(ece_after, 4)}   "
      f"(delta {round(ece_after - ece_before, 4)})")

## Recap / next steps

We ran the full reference pipeline offline:

1. **Load** the bundled gold QA set and two candidate models (`tools._common`).
2. **Score** each model with exact-match accuracy and token-F1
   (`tools.score.score_dataset`).
3. **Calibrate**: ECE, reliability bins, and the confident-wrong list
   (`tools.calibration.analyze`).
4. **Compare** the overconfident vs. calibrated model, and reproduced two of
   the README's experiments (threshold sweep, confidence edit -> ECE).

The takeaway: a model can look fine on accuracy yet be untrustworthy because it
is *confidently wrong*. ECE and the confident-wrong list expose that gap.

Next steps from the README to explore on your own:

- Add a new question to `data/gold.json` plus a matching prediction, then
  re-score.
- Edit a `confidence` in a candidate file on disk and re-run the CLI
  (`python -m tools.score ...` then `python -m tools.calibration ...`).
- In the full project, run the `/evaluate` command to have the agent write an
  honest markdown report to `reports/`.